# 1. Import Library
Memasukkan semua library yang dibutuhkan untuk tahap EDA, Preprocessing, Modeling, dan Evaluasi.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# 2. EDA & Preprocessing

### Load Dataset
Sesuai ketentuan, data **train** hanya digunakan untuk training (termasuk sebagai patokan imputasi dan scaling), dan data **test** khusus digunakan untuk evaluasi akhir. Tidak ada split data lagi di sini.

*(Catatan: Untuk menghindari RAM laptop penuh / Kernel Crash akibat memuat 7,5 juta baris sekaligus, kita langsung mensampling data saat awal load)*

In [2]:
# Mengambil data yang sudah di-split sebelumnya
train_df = pd.read_csv('../dataset/data/train.csv')
test_df = pd.read_csv('../dataset/data/test.csv')

# ================== MEMORY OPTIMIZATION ==================
# Agar laptop tidak crash (Out of Memory), kita ambil 
# 300.000 sampel acak untuk train, dan 50.000 untuk test.
train_limit = min(300000, len(train_df))
test_limit = min(50000, len(test_df))
train_df = train_df.sample(n=train_limit, random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=test_limit, random_state=42).reset_index(drop=True)
# =========================================================

print("Shape of train_df:", train_df.shape)
print("Shape of test_df:", test_df.shape)
train_df.head()

Shape of train_df: (300000, 46)
Shape of test_df: (50000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-4521088,Source1,0,2022-01-29 08:35:00,2022-01-29 09:57:46,41.451091,-75.582945,41.462787,-75.559777,1.446,...,False,False,False,False,False,False,Day,Day,Day,Day
1,A-5416285,Source1,0,2022-04-02 15:39:00.000000000,2022-04-02 16:56:48.000000000,38.576231,-121.466719,38.575688,-121.466980,0.040,...,False,False,False,False,False,False,Day,Day,Day,Day
2,A-5113203,Source1,0,2022-08-05 14:27:33,2022-08-05 19:24:32,26.188417,-80.152091,26.130809,-80.169481,4.124,...,False,False,False,False,False,False,Day,Day,Day,Day
3,A-4770690,Source1,0,2022-08-22 08:05:00,2022-08-22 09:20:00,28.326225,-81.401709,28.326219,-81.400713,0.061,...,False,False,False,False,False,False,Day,Day,Day,Day
4,A-7660663,Source1,1,2017-11-16 16:47:28,2017-11-16 22:50:28,42.342916,-88.169230,42.343890,-88.169230,0.067,...,False,False,False,False,False,False,Night,Day,Day,Day


### 2.1 Missing Value Handling
Penanganan missing value dilakukan dengan mengisi kekosongan (imputasi). Kolom numerik diisi dengan **median**, sedangkan kolom kategorikal diisi dengan **modus**. Ingat, perhitungan median/modus **hanya** didapatkan dari data train.

In [3]:
print("Missing values di data train:\n", train_df.isnull().sum())

# Tentukan mana kolom numerik dan kategorikal
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Simpan nilai imputasi untuk deployment
median_impute = {}
for col in numeric_cols:
    median_val = train_df[col].median()
    median_impute[col] = median_val if not pd.isna(median_val) else 0
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

mode_impute = {}
for col in categorical_cols:
    mode_val = train_df[col].mode()[0]
    mode_impute[col] = mode_val if not train_df[col].mode().empty else 'unknown'
    train_df[col] = train_df[col].fillna(mode_val)
    test_df[col] = test_df[col].fillna(mode_val)

Missing values di data train:
 ID                            0
Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  129015
End_Lng                  129015
Distance(mi)                  0
Description                   0
Street                      422
City                         17
County                        0
State                         0
Zipcode                      81
Country                       0
Timezone                    325
Airport_Code                889
Weather_Timestamp          4780
Temperature(F)             6475
Wind_Chill(F)             75899
Humidity(%)                6891
Pressure(in)               5540
Visibility(mi)             6999
Wind_Direction             6951
Wind_Speed(mph)           22019
Precipitation(in)         83777
Weather_Condition          6864
Amenity                       0
Bump     

### 2.2 Outliers Handling
Kita menggunakan metode IQR (Interquartile Range) untuk membatasi (capping) nilai outlier agar tidak merusak model. PENTING: Kolom target dilarang keras untuk dicapping!

In [4]:
iqr_bounds = {}
for col in numeric_cols:
    if col == 'Severity':
        continue
        
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_bounds[col] = {'lower': lower_bound, 'upper': upper_bound}
    
    train_df[col] = np.where(train_df[col] < lower_bound, lower_bound, train_df[col])
    train_df[col] = np.where(train_df[col] > upper_bound, upper_bound, train_df[col])
    
    test_df[col] = np.where(test_df[col] < lower_bound, lower_bound, test_df[col])
    test_df[col] = np.where(test_df[col] > upper_bound, upper_bound, test_df[col])

### 2.3 Encoding
Mengubah data string (teks) menjadi angka numerik yang bisa dipahami model menggunakan `LabelEncoder`. Dibuat secepat mungkin menggunakan dictionary mapping.

In [5]:
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    le_dict = dict(zip(le.classes_, range(len(le.classes_))))
    unknown_val = len(le.classes_)
    label_encoders[col] = {
        'classes': le.classes_.tolist(),
        'mapping': le_dict
    }
    
    le.classes_ = np.append(le.classes_, '<unknown>')
    
    test_df[col] = test_df[col].astype(str).map(le_dict).fillna(unknown_val).astype(int)

### 2.4 Transformasi (Scaling Fitur)
Melakukan standardisasi (rata-rata=0, std=1) agar model yang sensitif pada jarak bekerja lebih optimal.

In [6]:
# TARGET KOLOM DISET KE 'Severity'
TARGET_KOLOM = 'Severity' 

try:
    # Memisahkan Fitur (X) dan Target Label (y)
    X_train_raw = train_df.drop(columns=[TARGET_KOLOM])
    y_train = train_df[TARGET_KOLOM]
    
    X_test_raw = test_df.drop(columns=[TARGET_KOLOM])
    y_test = test_df[TARGET_KOLOM]
    
    # Inisiasi Scaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw) # Scaling test pakai patokan train
except KeyError:
    print("WARNING: Jangan lupa ubah variabel TARGET_KOLOM di atas dengan nama kolom yang benar di dataset ini.")

# 3. Modeling
Membuat model klasifikasi. Kita akan menggunakan algoritma **Random Forest Classifier** karena secara umum algoritma ini tangguh (robust) terhadap struktur data yang belum linear sempurna dan memberikan akurasi yang solid.

In [7]:
try:
    # Menggunakan class_weight='balanced' untuk mengatasi imbalanced data (kelas 1 dan 4 yang langka)
    # Karena di Langkah 2 data sudah di-sample, datanya sudah teracak dengan baik.
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
    
    print(f"Melatih model Random Forest (Balanced) pada seluruh train_df ({len(X_train_scaled)} baris)...")
    rf_model.fit(X_train_scaled, y_train)
    print("Model Random Forest berhasil dilatih!")
except NameError:
    print("Harap lengkapi tahap 2.4 terlebih dahulu.")

Melatih model Random Forest (Balanced) pada seluruh train_df (300000 baris)...
Model Random Forest berhasil dilatih!


# 4. Evaluasi (Test)
Menggunakan fitur dari data uji (test.csv) ke dalam model, lalu membandingkan hasil prediksinya dengan label aslinya untuk mencari nilai Recall, Precision, dan Accuracy.

In [8]:
try:
    print(f"Menguji data test...")
    y_pred = rf_model.predict(X_test_scaled)
    
    # Evaluasi Metriks
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print("=== HASIL EVALUASI MODEL ===")
    print(f"Akurasi   (Accuracy)  : {acc:.4f}")
    print(f"Presisi   (Precision) : {prec:.4f}")
    print(f"Recall    (Recall)    : {rec:.4f}")
    
    # Menampilkan report lebih lengkap (opsional)
    print("\nClassification Report Lengkap:\n")
    print(classification_report(y_test, y_pred, zero_division=0))
except NameError:
    print("Harap lengkapi tahap sebelumnya.")

Menguji data test...
=== HASIL EVALUASI MODEL ===
Akurasi   (Accuracy)  : 0.5066
Presisi   (Precision) : 0.6907
Recall    (Recall)    : 0.5066

Classification Report Lengkap:

              precision    recall  f1-score   support

           0       0.50      1.00      0.67     25077
           1       0.88      0.01      0.02     24923

    accuracy                           0.51     50000
   macro avg       0.69      0.51      0.35     50000
weighted avg       0.69      0.51      0.35     50000



# 5. Simpan Model & Artifacts
Menyimpan model yang sudah dilatih beserta preprocessing artifacts ke folder `model/` untuk digunakan di deployment.

In [ ]:
import joblib
import os

model_dir = '../model'
os.makedirs(model_dir, exist_ok=True)

feature_names = X_train_raw.columns.tolist()

preprocessing = {
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'target_column': TARGET_KOLOM,
    'median_impute': median_impute,
    'mode_impute': mode_impute,
    'iqr_bounds': iqr_bounds,
    'label_encoders': label_encoders,
    'feature_names': feature_names
}

joblib.dump(rf_model, os.path.join(model_dir, 'random_forest.pkl'))
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))
joblib.dump(preprocessing, os.path.join(model_dir, 'preprocessing.pkl'))

print(f"Model dan artifacts berhasil disimpan ke folder '{model_dir}/'")
print("  - random_forest.pkl")
print("  - scaler.pkl")
print("  - preprocessing.pkl")